# formal_30_v2: TopologyEditor manual validation

This entry preserves manual browser validation of the formal_30_v1 artifact. No repair is applied before manual inspection.

## Artifacts
- Vue: `data/datasets/components/03_complex/TopologyEditor/vue/TopologyEditor.vue`
- San: `data/experiments/pipeline/batches/formal_30_v1/complex/TopologyEditor.san`
- San SHA-256: `7ED750DFB6D3E1F674DC968E891C1581F7A821D33150B4D8652DFE90245DE898`

## Automatic formal_30_v1 result
The batch did not produce a repaired final artifact. Its repair request failed with HTTP 400 because the read operation timed out. Therefore, manual validation uses the initial generated San artifact and records the condition as an execution error, not as a completed automatic evaluation failure.

## Manual validation checklist
- Compare initial nodes, edges, selection, inspector, coordinates, connection counts, zoom, and event log.
- Select, rename, move, add, and delete nodes.
- Start, complete, and cancel connections.
- Change zoom and compare the canvas transform.
- Start and stop simulation and compare styles, states, and event log.

## Observed defects and formal_30_v2 repair
Manual validation found four visible problems: the title was empty, nodes did not move, connections were not created visibly, and simulation did not visibly update node state.

The title defect came from omitting the Vue prop default. The interaction defects shared a second root cause: generated handlers directly mutated objects and arrays returned by `this.data.get()` or computed `selectedNode`, such as `selectedNode.x = ...`, `edges.push(...)`, `nodes.push(...)`, and `node.status = ...`. These mutations bypassed San's reactive update path.

The v2 repair initializes the title and rewrites rename, movement, node addition, connection creation, and simulation ticks to create new objects or arrays and assign them through `this.data.set()`. The frozen v1 artifact remains unchanged.

- Fixed v2 San: `data/experiments/pipeline/batches/formal_30_v2/complex/TopologyEditor_repaired.san`
- First fixed v2 SHA-256: `AAC3E0EEBA9940CD2C64D2E3B01BF066F761C8F9F796B5BD12F4C4871B7C2C0E`

## Follow-up simulation timer defect
After the reactive-state repair, starting simulation still logged only `状态模拟已启动`; no periodic node-status messages appeared. The migrated timer was created only in `attached()` and stored in reactive view data, while `toggleSimulation()` only changed a flag and did not ensure that an active timer existed.

The follow-up repair stores the timer as a component instance resource. Starting simulation now performs one immediate `simulateTick()` and starts a 1.8-second interval; stopping simulation or disposing the component clears that interval. Log insertion also writes a new array through `this.data.set()`.

- Latest fixed v2 SHA-256: `685F92DE10FFFA27BF54B6E545E1B24DC1DD405C7370F5781D0EC5C0A2901F03`

Result: passed by user manual browser validation on `2026-10-22T14:34:21+08:00` after the title, reactive-state, and simulation-timer repairs.